# 07 — Explainability: SHAP (XGBoost) + Integrated Gradients (ConvLSTM)

**Week 7 goal:** Answer *why* each model makes each prediction.

This matters for two reasons:
1. **Trust** — a government / NGO needs to know *which signal* triggered a drought alert before acting on it.
2. **Debugging** — if the model is learning a spurious shortcut (e.g. using land cover as a proxy for soil moisture), explanations reveal it before deployment.

We use two complementary techniques:

| Method | Model | What it answers |
|--------|-------|-----------------|
| **SHAP** (SHapley Additive exPlanations) | XGBoost | Which feature pushed this prediction up or down? |
| **Integrated Gradients** | ConvLSTM | Which input pixels/channels mattered most? |

### SHAP in plain English
Imagine you're sharing credit for a prediction among 30 features. SHAP asks: if you added feature X to the model one-at-a-time in every possible order, how much did it move the prediction on average? That average is the Shapley value — a fair credit split from cooperative game theory.

For tree models (XGBoost), SHAP can be computed exactly in polynomial time — no approximation.

### Integrated Gradients in plain English
For a neural network, the gradient `∂output/∂input` tells you which input dimensions the output is *currently* most sensitive to. But raw gradients are noisy. Integrated Gradients averages gradients along the straight line from a *baseline* (all-zeros = no signal) to the actual input. The result is an attribution map with the same shape as the input — you can literally look at it as a spatial map.

## Prerequisites — install once, then restart the kernel

Run the cell below **once**. After it finishes, click **Restart** (VS Code: `Ctrl+Shift+P` → *Notebook: Restart Kernel*), then run all cells from the top again.

> **Why restart?** Python caches already-imported modules. If an old version of xgboost was loaded by Anaconda before the install ran, the new version won't be picked up until the kernel is fresh.

In [1]:
%pip install -q --upgrade xgboost shap captum
print('Done — restart the kernel now, then run all cells from the top.')

Note: you may need to restart the kernel to use updated packages.
Done — restart the kernel now, then run all cells from the top.


In [ ]:
import sys, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch

# Auto-detect Colab vs local — no manual changes needed
try:
    from google.colab import drive
    import subprocess, shutil, os
    drive.mount('/content/drive')
    _BASE = '/content/drive/MyDrive/botswana-drought-flood'
    os.makedirs(_BASE, exist_ok=True)

    # Always clone latest code from GitHub into /tmp (avoids Drive git-pull conflicts
    # and stale bytecode caches — Drive is used for data/ only).
    if os.path.exists('/tmp/bdf'):
        shutil.rmtree('/tmp/bdf')
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/andrew-simons/botswana-drought-flood',
                    '/tmp/bdf'], capture_output=True, check=True)
    for _f in ['src', 'notebooks', 'scripts', 'app', '.streamlit']:
        _dst = f'{_BASE}/{_f}'
        if os.path.exists(_dst):
            shutil.rmtree(_dst)
        shutil.copytree(f'/tmp/bdf/{_f}', _dst)
    _hash = subprocess.run(['git', 'log', '--oneline', '-1'], cwd='/tmp/bdf',
                           capture_output=True, text=True).stdout.strip()
    print(f'Code updated to: {_hash}')

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rioxarray'], check=True)
    sys.path.insert(0, f'{_BASE}/src')
    print('Ready.')
except ImportError:
    _BASE = '..'
    sys.path.insert(0, '../src')

CUBE = f'{_BASE}/data/cube'
CKPT = f'{_BASE}/data/cube/model_convlstm.pt'

from botswana_ds.train.metrics import batch_summary

dynamic = np.load(f'{CUBE}/dynamic.npy', mmap_mode='r')
static  = np.load(f'{CUBE}/static.npy')
labels  = np.load(f'{CUBE}/labels.npy',  mmap_mode='r')
mask    = np.load(f'{CUBE}/mask.npy')
meta    = json.load(open(f'{CUBE}/meta.json'))

T, C_dyn, H, W  = dynamic.shape
TRAIN_END        = meta['splits']['train_end_idx']
LABEL_NAMES      = meta['label_channels']
DYN_CHANNELS     = meta['dynamic_channels']
STATIC_CHANNELS  = meta['static_channels']

INPUT_LEN = 3   # XGBoost uses 3-month lags

print(f'Cube: T={T}, C_dyn={C_dyn}, H={H}, W={W}')
print(f'Train: 0–{TRAIN_END-1}  Test: {TRAIN_END}–{T-1}')
print(f'Dynamic channels : {DYN_CHANNELS}')
print(f'Static  channels : {STATIC_CHANNELS}')
print(f'CKPT: {CKPT}')

---
## Part 1 — SHAP feature importance for XGBoost

### Step 1a: Re-train XGBoost and keep the model objects

In notebook 05 we discarded the trained models after computing metrics. Here we keep them by passing `return_models=True`. Each of the 3 label channels gets its own `XGBRegressor`.

In [ ]:
# Mac: pre-load libomp so xgboost can find OpenMP
import ctypes, pathlib
_libomp = pathlib.Path('/opt/homebrew/opt/libomp/lib/libomp.dylib')
if _libomp.exists():
    ctypes.cdll.LoadLibrary(str(_libomp))

from botswana_ds.models.baselines import xgb_forecast

print('Training XGBoost (return_models=True)...')
xgb_preds, xgb_targets, xgb_models = xgb_forecast(
    dynamic,
    static,
    labels,
    mask,
    train_end=TRAIN_END,
    input_len=INPUT_LEN,
    max_train_rows=400_000,
    seed=42,
    return_models=True,
)
print(f'Got {len(xgb_models)} XGBRegressor objects (one per label channel)')

# Feature names for the 30-dimensional input vector:
#   [ch_lag1, ch_lag2, ch_lag3  for each of 9 channels]  +  [elev, slope, lc]
FEAT_NAMES = (
    [f'{ch}_lag{l}' for l in range(INPUT_LEN, 0, -1) for ch in DYN_CHANNELS]
    + STATIC_CHANNELS
)
print(f'Feature names ({len(FEAT_NAMES)}): {FEAT_NAMES[:6]} ... {FEAT_NAMES[-3:]}')


Training XGBoost (return_models=True)...


### Step 1b: Compute SHAP values

`shap.TreeExplainer` is the fast exact SHAP algorithm for tree models. It computes Shapley values analytically using the tree structure — no sampling.

We compute SHAP values on a subset of the test set (all valid pixels for the first 6 test months). This gives ~120K rows — enough for stable importance estimates without taking too long.

In [ ]:
import shap

valid_h, valid_w = np.where(mask)
n_valid = len(valid_h)

# Build a test-set feature matrix for the first 6 test months (faster than all 24)
N_EXPLAIN = 6
dyn_px   = np.asarray(dynamic[:, :, valid_h, valid_w])    # (T, C_dyn, n_valid)
stat_px  = static[:, valid_h, valid_w].T                  # (n_valid, C_stat)

X_explain_list = []
for ti in range(N_EXPLAIN):
    s = TRAIN_END - INPUT_LEN + ti    # window start so target = TRAIN_END + ti
    lag = dyn_px[s: s + INPUT_LEN].transpose(2, 0, 1).reshape(n_valid, -1)
    X_explain_list.append(np.concatenate([lag, stat_px], axis=1))

X_explain = np.concatenate(X_explain_list, axis=0).astype(np.float32)
print(f'Explaining {X_explain.shape[0]:,} rows ({N_EXPLAIN} months × {n_valid:,} pixels)')

# Compute SHAP values for each label channel
shap_values = {}  # label_name -> (n_rows, n_feats) array
for c, name in enumerate(LABEL_NAMES):
    explainer = shap.TreeExplainer(xgb_models[c])
    sv = explainer.shap_values(X_explain)
    shap_values[name] = sv
    print(f'  {name}: SHAP shape = {sv.shape}')

### Step 1c: Feature importance bar chart

The bar chart shows **mean |SHAP value|** per feature — the average absolute impact on the model output across all explained rows. This is the most interpretable summary: a tall bar means that feature consistently moves predictions up or down.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 7))

for ax, name in zip(axes, LABEL_NAMES):
    mean_abs = np.abs(shap_values[name]).mean(axis=0)   # (n_feats,)
    order    = np.argsort(mean_abs)[::-1][:15]           # top-15

    ax.barh(
        [FEAT_NAMES[i] for i in order[::-1]],
        mean_abs[order[::-1]],
        color='steelblue'
    )
    ax.set_xlabel('Mean |SHAP value|', fontsize=9)
    ax.set_title(f'{name}\ntop-15 features', fontsize=10)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='x', alpha=0.3)

fig.suptitle('XGBoost SHAP feature importance (mean |SHAP|)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{CUBE}/07_shap_bar.png', dpi=120, bbox_inches='tight')
plt.show()

# Print summary for each label
for name in LABEL_NAMES:
    mean_abs = np.abs(shap_values[name]).mean(axis=0)
    top = sorted(zip(FEAT_NAMES, mean_abs), key=lambda x: -x[1])[:5]
    print(f'{name}: top-5 = {[(f, round(v, 3)) for f, v in top]}')

### Step 1d: SHAP beeswarm plot

The beeswarm plot is richer than the bar chart: each dot is one row (pixel × month), colour-coded by the **feature value** (red = high, blue = low). This lets you see directionality:
- Red dot (high precipitation) with positive SHAP → high rainfall pushes the prediction away from drought (expected for SPI-3).
- Blue dot (low NDVI, e.g. bare sand) with negative SHAP → sparse vegetation pushes towards drought-like anomaly.

In [ ]:
# shap.summary_plot renders in-place — one plot per label channel
for name, c in zip(LABEL_NAMES, range(3)):
    print(f'\n── Beeswarm: {name} ──')
    shap.summary_plot(
        shap_values[name],
        X_explain,
        feature_names=FEAT_NAMES,
        max_display=15,
        show=False,
        plot_size=(8, 5),
    )
    plt.title(f'SHAP beeswarm — {name}', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'{CUBE}/07_shap_beeswarm_{name}.png', dpi=120, bbox_inches='tight')
    plt.show()

---
## Part 2 — Integrated Gradients for ConvLSTM

SHAP works on tree models because their prediction function can be analytically decomposed. For neural networks, we need gradient-based attribution.

**Why not just use raw gradients?**  
Raw gradients (`∂output/∂input`) tell you the local slope, not the total contribution. A feature can have zero gradient yet still matter a lot (e.g., a saturated sigmoid). Integrated Gradients (Sundararajan et al. 2017) fixes this by averaging gradients along a straight interpolation path from a *baseline* input to the actual input.

**The formula:**
```
IG(x)_i = (x_i - x'_i) × ∫₀¹ (∂F(x' + α(x - x'))/∂x_i) dα
```
where `x'` is the baseline and `α` sweeps from 0 → 1 in `n_steps` steps.

We use [Captum](https://captum.ai/) — PyTorch's official interpretability library — which implements this cleanly.

### Step 2a: Load the ConvLSTM checkpoint

In [ ]:
# captum is already installed from the XGBoost cell above

import os

from botswana_ds.models.convlstm import ConvLSTMForecaster
from botswana_ds.data.dataset import BotswanaCube, compute_norm_stats

if not os.path.exists(CKPT):
    raise FileNotFoundError(
        f'Checkpoint not found at {CKPT}.\n'
        'Run notebook 06 (ConvLSTM training) first, then copy the .pt file here.'
    )

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {device}')

ckpt = torch.load(CKPT, map_location=device)
print(f'Checkpoint: epoch={ckpt["epoch"]}  val_MAE={ckpt["val_loss"]:.4f}')

# Rebuild model with same architecture as notebook 06
IN_CH     = C_dyn + 3      # 9 dynamic + 3 lagged labels
STATIC_CH = static.shape[0]
HORIZON   = 1

model = ConvLSTMForecaster(
    in_ch=IN_CH, hidden_ch=64, n_targets=3,
    horizon=HORIZON, static_ch=STATIC_CH
).to(device)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print('ConvLSTM loaded.')


### Step 2b: Wrap model for Captum

Captum's `IntegratedGradients` expects a callable `f(input_tensor) -> scalar_output`. Our ConvLSTM takes `(x, static)` and returns a spatial map `(B, horizon, 3, H, W)`. We need a thin wrapper that:
1. Accepts only `x` as the differentiable input (we'll keep static fixed)
2. Returns a single-channel spatial output (one label, one forecast step) summed over the spatial grid → a scalar

We attribute w.r.t. `x` (the dynamic input sequence), which has shape `(B, L, C, H, W)`.

In [ ]:
from captum.attr import IntegratedGradients

# Load norm stats so we can build a properly normalised input sample
norm_stats = compute_norm_stats(CUBE, train_slice=(0, TRAIN_END))

# Load one test window from the Dataset
INPUT_LEN_CONVLSTM = 24   # must match notebook 06 training
test_ds = BotswanaCube(
    CUBE, input_len=INPUT_LEN_CONVLSTM, horizon=HORIZON,
    time_slice=(TRAIN_END - INPUT_LEN_CONVLSTM, T), norm_stats=norm_stats
)
# Use the first test window (Jan 2022 → predicts Feb–Apr 2022)
sample = test_ds[0]

x_in     = sample['x'].unsqueeze(0).to(device)        # (1, L, C, H, W)
static_in = sample['static'].unsqueeze(0).to(device)   # (1, Cs, H, W)
y_true    = sample['y']                                # (horizon, 3, H, W)

print(f'Input x shape : {x_in.shape}')
print(f'Static shape  : {static_in.shape}')
print(f'Target shape  : {y_true.shape}')

# Pre-build mask tensor once (reused inside forward on every IG step)
mask_t = torch.tensor(mask, device=device, dtype=torch.float32)

# Wrapper: attribute wrt x only; target = sum of pred over spatial mask
def forward_single(x, label_ch=0, horizon_step=0):
    """Handles any batch size — Captum passes B>1 when internal_batch_size>1."""
    B = x.shape[0]
    static_b = static_in.expand(B, -1, -1, -1)             # (B, Cs, H, W)
    pred = model(x, static_b)                               # (B, horizon, 3, H, W)
    pred_ch = pred[:, horizon_step, label_ch]               # (B, H, W)
    return (pred_ch * mask_t).sum(dim=(1, 2))               # (B,)

# Quick forward check
with torch.no_grad():
    out = forward_single(x_in)
print(f'Forward check scalar output: {out.item():.4f}')


### Step 2c: Compute Integrated Gradients

We'll compute attributions for all 3 label channels predicting 1 step ahead. The baseline is an all-zeros input ("no signal" — normal conditions). `n_steps=50` is the standard accuracy/speed trade-off.

In [ ]:
baseline = torch.zeros_like(x_in)   # "no signal" baseline

ig_attrs = {}  # label_name -> (L, C, H, W) attribution array

for c, name in enumerate(LABEL_NAMES):
    ig = IntegratedGradients(lambda x: forward_single(x, label_ch=c, horizon_step=0))
    attrs, delta = ig.attribute(
        x_in,
        baselines=baseline,
        n_steps=20,           # lower = less memory; 20 is enough for attribution
        internal_batch_size=4, # stream 4 steps at a time to avoid OOM on T4
        return_convergence_delta=True,
    )
    # attrs: (1, L, C, H, W) — squeeze batch dim
    ig_attrs[name] = attrs.squeeze(0).detach().cpu().numpy()   # (L, C, H, W)
    print(f'  {name}: shape={ig_attrs[name].shape}  convergence_delta={delta.item():.4f}')

# Sum attribution over timesteps → (C, H, W): which channel drives the prediction?
ig_channel_mean = {
    name: np.abs(ig_attrs[name]).sum(axis=0)   # sum |attr| over L timesteps
    for name in LABEL_NAMES
}
print('\nDone. Channel-summed attribution shape:', next(iter(ig_channel_mean.values())).shape)


### Step 2d: Spatial attribution maps

We visualise two views:
1. **Per-channel global importance** — bar chart of mean |attribution| summed over the full spatial grid per input channel, then averaged across timesteps. Shows *which variables* drive the ConvLSTM.
2. **Spatial maps** — for the top-3 most important input channels, show where on the map the model pays most attention.

In [ ]:
# Channel names = dynamic channels + lagged label channels
ALL_CHANNEL_NAMES = DYN_CHANNELS + [f'{n}_lag' for n in LABEL_NAMES]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, name in zip(axes, LABEL_NAMES):
    # ig_attrs[name]: (L, C, H, W) — sum over spatial & timestep, keep channel dim
    ch_importance = np.abs(ig_attrs[name]).sum(axis=(0, 2, 3))  # (C,)
    order = np.argsort(ch_importance)[::-1]

    ax.barh(
        [ALL_CHANNEL_NAMES[i] for i in order[::-1]],
        ch_importance[order[::-1]],
        color='coral'
    )
    ax.set_xlabel('Summed |attribution|', fontsize=9)
    ax.set_title(f'{name}\nchannel importance', fontsize=10)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='x', alpha=0.3)

fig.suptitle('ConvLSTM Integrated Gradients — per-channel importance', fontsize=12)
plt.tight_layout()
plt.savefig(f'{CUBE}/07_ig_channel_importance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Spatial attribution: sum |attr| over all channels and timesteps → (H, W)
# Shows WHERE the model pays attention in the grid.

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for col, name in enumerate(LABEL_NAMES):
    # (L, C, H, W) → (H, W)
    spatial_total = np.abs(ig_attrs[name]).sum(axis=(0, 1))
    spatial_total = np.where(mask, spatial_total, np.nan)

    # Row 0: attribution intensity map
    ax0 = axes[0, col]
    im0 = ax0.imshow(spatial_total, cmap='hot_r', origin='upper')
    ax0.set_title(f'{name}\n|attribution| spatial sum', fontsize=9)
    ax0.axis('off')
    plt.colorbar(im0, ax=ax0, fraction=0.03)

    # Row 1: ground-truth label for the same month (to compare regions)
    gt = np.where(mask, labels[TRAIN_END, col], np.nan)
    ax1 = axes[1, col]
    im1 = ax1.imshow(gt, cmap='RdYlBu', vmin=-2.5, vmax=2.5, origin='upper')
    ax1.set_title(f'{name}\nactual label (Jan 2022)', fontsize=9)
    ax1.axis('off')
    plt.colorbar(im1, ax=ax1, fraction=0.03)

fig.suptitle('ConvLSTM attribution maps vs. ground truth — Jan 2022 (first test month)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{CUBE}/07_ig_spatial_maps.png', dpi=120, bbox_inches='tight')
plt.show()

print('Interpretation guide:')
print('  Bright areas in the top row = pixels the ConvLSTM relied on most for its prediction.')
print('  If bright areas align with drought regions in the bottom row, the model is attending')
print('  to the right places (not just coastlines/borders).')

### Step 2e: Attribution over time

The ConvLSTM sees 24 months of history. How far back does it look? Plot the mean |attribution| summed over channels and pixels, per timestep. A spike near the end means the model mostly uses recent months; a flat curve means it weighs all history equally.

In [ ]:
valid_h, valid_w = np.where(mask)

fig, ax = plt.subplots(figsize=(10, 4))

for name in LABEL_NAMES:
    # ig_attrs[name]: (L, C, H, W)
    temporal = np.abs(ig_attrs[name])[:, :, valid_h, valid_w].sum(axis=(1, 2))  # (L,)
    ax.plot(range(-INPUT_LEN_CONVLSTM, 0), temporal, marker='o', ms=3, label=name)

ax.set_xlabel('Timestep relative to forecast (0 = most recent input month)', fontsize=9)
ax.set_ylabel('Summed |attribution|', fontsize=9)
ax.set_title('ConvLSTM: attribution weight over input sequence', fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CUBE}/07_ig_temporal.png', dpi=120, bbox_inches='tight')
plt.show()

for name in LABEL_NAMES:
    temporal = np.abs(ig_attrs[name])[:, :, valid_h, valid_w].sum(axis=(1, 2))
    recent_3 = temporal[-3:].sum() / temporal.sum()
    print(f'  {name}: {100*recent_3:.1f}% of attribution from the last 3 months')

---
## Part 3 — Cross-model comparison

Now we have two independent views of feature importance:
- **SHAP** (XGBoost): tells us which of the 30 tabular features drive the flat, per-pixel predictions.
- **IG** (ConvLSTM): tells us which of the 12 input channels drive the spatial predictions.

When both agree (e.g., CHIRPS precipitation is top-1 for SPI-3 in both), that's a strong signal the feature genuinely matters — not a model artefact.

When they disagree, investigate: the ConvLSTM may be using spatial context that XGBoost can't access, or one model may be overfitting to a spurious correlation.

In [ ]:
print('═' * 60)
print('Feature importance comparison: SHAP (XGBoost) vs IG (ConvLSTM)')
print('═' * 60)

for name in LABEL_NAMES:
    print(f'\n── {name} ──')

    # SHAP top-5 (aggregate lag variants by channel)
    shap_mean = np.abs(shap_values[name]).mean(axis=0)
    # Group by base channel (sum lags)
    ch_shap = {ch: 0.0 for ch in DYN_CHANNELS + STATIC_CHANNELS}
    for fi, feat in enumerate(FEAT_NAMES):
        base = feat.rsplit('_lag', 1)[0] if '_lag' in feat else feat
        if base in ch_shap:
            ch_shap[base] += shap_mean[fi]
    shap_top5 = sorted(ch_shap.items(), key=lambda x: -x[1])[:5]

    # IG top-5 (channel dimension)
    ig_ch = np.abs(ig_attrs[name]).sum(axis=(0, 2, 3))   # (C,) summed over L, H, W
    ig_top5 = sorted(zip(ALL_CHANNEL_NAMES, ig_ch), key=lambda x: -x[1])[:5]

    print(f'  SHAP top-5 : {[(f, round(v, 3)) for f, v in shap_top5]}')
    print(f'  IG   top-5 : {[(f, round(v, 2)) for f, v in ig_top5]}')

    # Agreement: channels in both top-5
    shap_set = {f for f, _ in shap_top5}
    ig_set   = {f for f, _ in ig_top5}
    overlap  = shap_set & ig_set
    print(f'  Agreement  : {overlap if overlap else "none — investigate!"}')

---
## Week 7 success criteria

| Check | Expected result |
|-------|-----------------|
| SHAP values computed on ≥6 test months | ✓ |
| CHIRPS precipitation in top-3 for SPI-3 | Expected — precipitation *is* SPI-3 |
| NDVI_lag in top-3 for ndvi_anom | Expected — lagged vegetation predicts future vegetation |
| SMAP SM in top-3 for sm_anom | Expected — lagged soil moisture predicts future SM |
| Spatial attribution aligns with drought region (not grid boundary) | Key sanity check |
| ≥2 channels agree between SHAP and IG for each label | Cross-model validation |
| IG temporal plot shows recency bias | Expected — recent months more informative |

### What comes next (Week 8)

1. **Write-up** — 2-3 page technical summary covering: data pipeline, model architecture, benchmark table, explanation findings.
2. **Streamlit demo** — interactive map showing current drought risk + SHAP feature bars on click.
3. **FastAPI skeleton** — `GET /risk`, `GET /explain` endpoints wrapping the ConvLSTM + XGBoost.

The explanation module you built here plugs directly into the `/explain` endpoint: pass a lat/lon, find the pixel, run IG, return the attribution dict as JSON.